In [1]:
import sys
import os
import whisper
import sqlite3
import json 
import pygame  # Added Pygame
from PyQt6.QtWidgets import (QApplication, QMainWindow, QPushButton, QVBoxLayout, 
                             QHBoxLayout, QWidget, QLabel, QFileDialog, QTextEdit, QFrame, QComboBox)
from PyQt6.QtCore import Qt, QThread, pyqtSignal, QUrl, QTimer # Added QTimer
# from PyQt6.QtMultimedia import QMediaPlayer, QAudioOutput # Removed QtMultimedia
from deep_translator import GoogleTranslator

# Hide pygame support prompt
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "hide"

class TranscriptionWorker(QThread):
    finished = pyqtSignal(dict) 
    error = pyqtSignal(str)

    def __init__(self, model, file_path):
        super().__init__()
        self.model = model
        self.file_path = file_path

    def run(self):
        try:
            result = self.model.transcribe(self.file_path)
            self.finished.emit(result)
        except Exception as e:
            self.error.emit(str(e))

class WhisperApp(QMainWindow):
    def __init__(self):
        super().__init__()
        
        # --- AI Initialization ---
        print("Loading Whisper 'base' model...")
        self.model = whisper.load_model("base")
        self.current_file = None
        self.current_timecodes = []
        self.is_paused = False # New state tracker for Pygame

        # --- UI Setup ---
        self.setWindowTitle("AI Lyrics Studio")
        self.setMinimumSize(1000, 700)
        
        self.central_widget = QWidget()
        self.setCentralWidget(self.central_widget)
        self.main_layout = QHBoxLayout(self.central_widget)

        self.sidebar = QFrame()
        self.sidebar.setFixedWidth(260)
        self.sidebar.setStyleSheet("background-color: #2c3e50; border-radius: 10px;")
        self.sidebar_layout = QVBoxLayout(self.sidebar)

        btn_css = """
            QPushButton { 
                background-color: #34495e; 
                color: white; 
                padding: 12px; 
                border: none; 
                border-radius: 5px;
                text-align: left; 
            }
            QPushButton:hover { background-color: #4e6a85; }
            QPushButton:disabled { background-color: #1a252f; color: #7f8c8d; }
        """

        self.btn_import = QPushButton("📁 Import Audio")
        self.btn_transcribe = QPushButton("⚡ Start AI")
        self.btn_transcribe.setEnabled(False)
        self.btn_play = QPushButton("▶ Play Audio")
        self.btn_play.setEnabled(False)
        self.btn_save = QPushButton("💾 Save Lyrics")
        self.btn_save.setEnabled(False)
        self.btn_clear = QPushButton("🗑️ Clear Workspace")

        for btn in [self.btn_import, self.btn_transcribe, self.btn_play, self.btn_save, self.btn_clear]:
            btn.setStyleSheet(btn_css)

        self.lang_label = QLabel("Target Language:")
        self.lang_label.setStyleSheet("color: white; margin-top: 15px; font-weight: bold;")
        
        self.combo_lang = QComboBox()
        self.combo_lang.addItems(["English", "French", "Spanish", "German", "Italian"])
        self.combo_lang.setStyleSheet("background-color: #1a1a1a; color: white; padding: 8px; border-radius: 5px;")

        self.sidebar_layout.addWidget(self.btn_import)
        self.sidebar_layout.addWidget(self.btn_transcribe)
        self.sidebar_layout.addWidget(self.lang_label)
        self.sidebar_layout.addWidget(self.combo_lang)
        self.sidebar_layout.addStretch()
        self.sidebar_layout.addWidget(self.btn_play)
        self.sidebar_layout.addWidget(self.btn_save)
        self.sidebar_layout.addWidget(self.btn_clear)

        self.workspace = QVBoxLayout()
        self.status_label = QLabel("Ready. Please import an audio file.")
        self.status_label.setStyleSheet("font-size: 14px; color: #34495e; font-weight: bold;")
        
        self.text_editor = QTextEdit()
        self.text_editor.setPlaceholderText("Transcription and translation results will appear here...")
        self.text_editor.setStyleSheet("font-size: 13px; padding: 10px; border: 1px solid #bdc3c7;")
        
        self.workspace.addWidget(self.status_label)
        self.workspace.addWidget(self.text_editor)

        self.main_layout.addWidget(self.sidebar)
        self.main_layout.addLayout(self.workspace)

        # --- Logic & Hardware Init ---
        self.init_db()
        self.init_multimedia()
        
        # Connections
        self.btn_import.clicked.connect(self.select_audio)
        self.btn_transcribe.clicked.connect(self.start_task)
        self.btn_play.clicked.connect(self.toggle_playback)
        self.btn_save.clicked.connect(self.save_lyrics)
        self.btn_clear.clicked.connect(self.clear_workspace)

        self.lang_map = {"English": "en", "French": "fr", "Spanish": "es", "German": "de", "Italian": "it"}

    def init_db(self):
        with sqlite3.connect("lyrics_studio.db") as conn:
            conn.execute('''CREATE TABLE IF NOT EXISTS saved_lyrics (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                filename TEXT, 
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                original_text TEXT, 
                translated_text TEXT, 
                language TEXT, 
                timecodes TEXT)''')

    def init_multimedia(self):
        """Sets up Pygame mixer instead of QtMultimedia."""
        pygame.mixer.init()
        # Timer to replace the missing positionChanged signal
        self.sync_timer = QTimer()
        self.sync_timer.timeout.connect(self.sync_lyrics_bridge)

    def select_audio(self):
        path, _ = QFileDialog.getOpenFileName(self, "Select Audio", "", "Audio Files (*.mp3 *.wav *.m4a)")
        if path:
            self.current_file = path
            pygame.mixer.music.stop() # Ensure previous song stops
            pygame.mixer.music.load(path) 
            self.status_label.setText(f"Loaded: {os.path.basename(path)}")
            self.btn_transcribe.setEnabled(True)
            self.btn_play.setEnabled(True)
            self.btn_play.setText("▶ Play Audio")
            self.is_paused = False

    def start_task(self):
        if not self.current_file: return
        self.status_label.setText("AI is transcribing... Please wait.")
        self.btn_transcribe.setEnabled(False)
        self.worker = TranscriptionWorker(self.model, self.current_file)
        self.worker.finished.connect(self.handle_result)
        self.worker.error.connect(self.handle_error)
        self.worker.start()

    def handle_result(self, result_dict):
        original_text = result_dict["text"].strip()
        self.current_timecodes = result_dict["segments"]
        target_lang = self.combo_lang.currentText()
        target_code = self.lang_map.get(target_lang, "en")
        display_text = original_text

        if target_code != "en":
            try:
                display_text = GoogleTranslator(source='auto', target=target_code).translate(original_text)
            except Exception: pass

        self.text_editor.setPlainText(display_text)
        self.btn_transcribe.setEnabled(True)
        self.btn_save.setEnabled(True)
        self.status_label.setText("Transcription Complete!")
        self.save_to_db(original_text, display_text, target_lang, json.dumps(self.current_timecodes))

    def save_to_db(self, original, translated, lang, timecodes_json):
        with sqlite3.connect("lyrics_studio.db") as conn:
            conn.execute("INSERT INTO saved_lyrics (filename, original_text, translated_text, language, timecodes) VALUES (?, ?, ?, ?, ?)",
                         (os.path.basename(self.current_file), original, translated, lang, timecodes_json))

    def toggle_playback(self):
        if not self.current_file: return
        
        # Logic to handle Play/Pause in Pygame
        if pygame.mixer.music.get_busy() and not self.is_paused:
            pygame.mixer.music.pause()
            self.is_paused = True
            self.btn_play.setText("▶ Play Audio")
            self.sync_timer.stop()
        else:
            if not pygame.mixer.music.get_busy(): # If music stopped or finished
                pygame.mixer.music.play()
            else:
                pygame.mixer.music.unpause()
            
            self.is_paused = False
            self.btn_play.setText("Pause")
            self.sync_timer.start(100) # Sync lyrics every 100ms

    def sync_lyrics_bridge(self):
        # Converts Pygame's ms to the position format the original sync_lyrics expects
        pos = pygame.mixer.music.get_pos()
        if pos != -1:
            self.sync_lyrics(pos)

    def sync_lyrics(self, position):
        seconds = position / 1000
        for segment in self.current_timecodes:
            if segment['start'] <= seconds <= segment['end']:
                print(f"Current segment: {segment['text']}")

    def handle_error(self, message):
        self.status_label.setText(f"AI Error: {message}")
        self.btn_transcribe.setEnabled(True)

    def clear_workspace(self):
        pygame.mixer.music.stop()
        self.sync_timer.stop()
        self.text_editor.clear()
        self.status_label.setText("Workspace cleared.")
        self.current_timecodes = []

    def save_lyrics(self):
        text = self.text_editor.toPlainText()
        if text:
            path, _ = QFileDialog.getSaveFileName(self, "Save Lyrics", "", "Text Files (*.txt)")
            if path:
                with open(path, "w", encoding="utf-8") as f: f.write(text)

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = WhisperApp()
    window.show()
    sys.exit(app.exec())

C:\Users\assyl\anaconda3\envs\whisper_app\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.10.20)
Hello from the pygame community. https://www.pygame.org/contribute.html
Loading Whisper 'base' model...


C:\Users\assyl\anaconda3\envs\whisper_app\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current segment:  One's upon the time there was a fairy I met her near the forest at dawn
Current se

SystemExit: 0

C:\Users\assyl\anaconda3\envs\whisper_app\lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
